In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

cwd = os.getcwd()
root = cwd.split("BernsteinMartingaleNet")[0] + "BernsteinMartingaleNet"
if root not in sys.path:
    sys.path.append(root)

from lib.utils import get_sequence_data, train_model
from lib.BLogistic import BLogistic
from lib.DistHead import NormalHead, StudentTHead, SkewedStudentTHead
from lib.Michenkow import Michenkow

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

folder_path = root + r"/MarketData/historical_data"
context_window = 60
X, Y     = get_sequence_data(folder_path, context_window)
dof      = 16
simple_X = torch.tensor(X[:, :, 0], device=device)
simple_Y = torch.tensor(Y, device=device).reshape(-1, 1)

dev_size = 40000
test_size = 40000
np.random.seed(0)
indices = np.random.permutation(simple_X.shape[0])

dev_indices = indices[:dev_size]
test_indices = indices[dev_size:dev_size + test_size]
train_indices = indices[dev_size + test_size:]
train_X = simple_X[train_indices]
train_Y = simple_Y[train_indices]
dev_X = simple_X[dev_indices, :]
dev_Y = simple_Y[dev_indices]
test_X = simple_X[test_indices, :]
test_Y = simple_Y[test_indices]

std = train_Y.std()
train_X = train_X / std
train_Y = train_Y / std
dev_X = dev_X / std
dev_Y = dev_Y / std
test_X = test_X / std
test_Y = test_Y / std
print("train_X", train_X.shape, "train_Y", train_Y.shape, "dev_X", dev_X.shape, "dev_Y", dev_Y.shape)
print("std", std, train_X.std())

using cached data from /home/jkim/coding_projects/BernsteinMartingaleNet/MarketData/historical_data/spy_1min_data_context_60.npz
train_X torch.Size([332426, 60]) train_Y torch.Size([332426, 1]) dev_X torch.Size([40000, 60]) dev_Y torch.Size([40000, 1])
std tensor(0.0004, device='cuda:0') tensor(1.0263, device='cuda:0')


In [2]:

layer_sizes = [128, 64, 32]
#layer_sizes = [8, 8, 8] # small testing size

class LSTMProbNNDist(nn.Module):
    def __init__(self, context_window, dist_head, device):
        super().__init__()
        self.context_window = context_window
        self.dist_head      = dist_head
        self.device         = device

        print(f"\nInitializing LSTM with context_window={context_window}, dist={dist_head.__class__.__name__}, dof={dist_head.num_params()}")

        # LSTM layers (3 layers with decreasing neurons: 128 -> 64 -> 32)
        self.lstm1 = nn.LSTM(
            input_size=1,
            hidden_size=layer_sizes[0],
            num_layers=1,
            batch_first=True,
            dropout=0
        )

        self.lstm2 = nn.LSTM(
            input_size=layer_sizes[0],
            hidden_size=layer_sizes[1],
            num_layers=1,
            batch_first=True,
            dropout=0
        )

        self.lstm3 = nn.LSTM(
            input_size=layer_sizes[1],
            hidden_size=layer_sizes[2],
            num_layers=1,
            batch_first=True,
            dropout=0
        )

        self.dropout = nn.Dropout(0.02)

        self.fc = nn.Linear(layer_sizes[2], dist_head.num_params())
        nn.init.uniform_(self.fc.weight, -0.01, 0.01)
        nn.init.zeros_(self.fc.bias)


    def forward(self, x, y):
        params = self.get_params(x)
        logpdf = self.dist_head.logpdf(
            y, params
        )
        return -logpdf.mean()

    def get_params(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(-1)
        elif x.dim() == 3 and x.shape[1] == 1:
            x = x.transpose(1, 2)

        x, _ = self.lstm1(x)
        x = self.dropout(x)

        x, _ = self.lstm2(x)
        x = self.dropout(x)

        x, _ = self.lstm3(x)
        x = self.dropout(x)

        x = x[:, -1, :]

        params = self.fc(x)
        return params

    def get_logpdf(self, x, sample_xs):
        params = self.get_params(x)
        return self.dist_head.logpdf(sample_xs, params)

    def get_pdf(self, x, sample_xs):
        return torch.exp(self.get_logpdf(x, sample_xs))

In [4]:
lr = 0.002
decay_step = 50
decay_gamma = 0.5
weight_decay = 0
num_steps = 1
batch_size = 512 * 8

In [5]:
# with lr = 0.002
model = LSTMProbNNDist(context_window, NormalHead(), device)
train_path = "Train_Normal_test/"
model, train_losses, dev_losses = train_model(model, train_X, train_Y, dev_X, dev_Y, lr, weight_decay, num_steps, batch_size=batch_size, device=device, output_folder=train_path)


Initializing LSTM with context_window=60, dist=NormalHead, dof=2
num batches 81
Init, Train Loss: 2.0901, Dev Loss: 2.0926, Dev Loss confidence interval: 1.9799, 2.2052
Step 0, Train Loss: 1.9176, Dev Loss: 1.9192, Dev Loss confidence interval: 1.8598, 1.9786, LR: 0.002000


In [4]:
model = LSTMProbNNDist(context_window, NormalHead(), device)
model.load_state_dict(torch.load("Train_Normal/model_80.pth"))#load
train_path = "Train_Normal_Warm/"
model, train_losses, dev_losses = train_model(model, train_X, train_Y, dev_X, dev_Y, 1e-4, weight_decay, 100, batch_size=batch_size, device=device, output_folder=train_path)


Initializing LSTM with context_window=60, dist=NormalHead, dof=2
num batches 81
Init, Train Loss: 1.9163, Dev Loss: 1.9178, Dev Loss confidence interval: 1.8630, 1.9726
Step 0, Train Loss: 1.9162, Dev Loss: 1.9178, Dev Loss confidence interval: 1.8625, 1.9730
Step 10, Train Loss: 1.9162, Dev Loss: 1.9178, Dev Loss confidence interval: 1.8624, 1.9732
Step 20, Train Loss: 1.9162, Dev Loss: 1.9178, Dev Loss confidence interval: 1.8623, 1.9732
Step 30, Train Loss: 1.9162, Dev Loss: 1.9178, Dev Loss confidence interval: 1.8624, 1.9732
Step 40, Train Loss: 1.9161, Dev Loss: 1.9178, Dev Loss confidence interval: 1.8624, 1.9733
Step 50, Train Loss: 1.9160, Dev Loss: 1.9179, Dev Loss confidence interval: 1.8624, 1.9734
Step 60, Train Loss: 1.9158, Dev Loss: 1.9179, Dev Loss confidence interval: 1.8624, 1.9734
Step 70, Train Loss: 1.9155, Dev Loss: 1.9179, Dev Loss confidence interval: 1.8625, 1.9733
Step 80, Train Loss: 1.9146, Dev Loss: 1.9180, Dev Loss confidence interval: 1.8625, 1.9735
Ste

In [ ]:
model = LSTMProbNNDist(context_window, NormalHead(), device)
model.load_state_dict(torch.load("Train_Normal_Warm/model_90.pth"))#load
train_path = "Train_Normal_Warmer/"
model, train_losses, dev_losses = train_model(model, train_X, train_Y, dev_X, dev_Y, 1e-4, weight_decay, 140, batch_size=batch_size, device=device, output_folder=train_path)


Initializing LSTM with context_window=60, dist=NormalHead, dof=2
num batches 81
Init, Train Loss: 1.9128, Dev Loss: 1.9170, Dev Loss confidence interval: 1.8606, 1.9734
Step 0, Train Loss: 1.9127, Dev Loss: 1.9169, Dev Loss confidence interval: 1.8606, 1.9732
Step 10, Train Loss: 1.9118, Dev Loss: 1.9163, Dev Loss confidence interval: 1.8605, 1.9721
Step 20, Train Loss: 1.9101, Dev Loss: 1.9196, Dev Loss confidence interval: 1.8635, 1.9756
Step 30, Train Loss: 1.9059, Dev Loss: 1.9394, Dev Loss confidence interval: 1.8839, 1.9949
Step 40, Train Loss: 1.8961, Dev Loss: 1.9535, Dev Loss confidence interval: 1.9001, 2.0068


In [ ]:
# with lr = 0.01
model = LSTMProbNNDist(context_window, NormalHead(), device)
train_path = "Train_Normal/"
model, train_losses, dev_losses = train_model(model, train_X, train_Y, dev_X, dev_Y, lr, weight_decay, num_steps, batch_size=batch_size, device=device, output_folder=train_path)


Initializing LSTM with context_window=60, dist=NormalHead, dof=2
num batches 81
Init, Train Loss: 2.0908, Dev Loss: 2.0933, Dev Loss confidence interval: 1.9805, 2.2061
Step 0, Train Loss: 1.9164, Dev Loss: 1.9180, Dev Loss confidence interval: 1.8613, 1.9746
Step 10, Train Loss: 1.9206, Dev Loss: 1.9223, Dev Loss confidence interval: 1.8596, 1.9851
Step 20, Train Loss: 1.9211, Dev Loss: 1.9228, Dev Loss confidence interval: 1.8597, 1.9860
Step 30, Train Loss: 1.9205, Dev Loss: 1.9222, Dev Loss confidence interval: 1.8596, 1.9849
Step 40, Train Loss: 1.9164, Dev Loss: 1.9180, Dev Loss confidence interval: 1.8611, 1.9750


KeyboardInterrupt: 

In [6]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect() 

In [4]:
model = LSTMProbNNDist(context_window, StudentTHead(), device)
train_path = "Train_StudentT/"
model, train_losses, dev_losses = train_model(model, train_X, train_Y, dev_X, dev_Y, lr, weight_decay, num_steps, batch_size=batch_size,
    device=device, output_folder=train_path, lr_decay_step=decay_step, lr_decay_gamma=decay_gamma)


Initializing LSTM with context_window=60, dist=StudentTHead, dof=3
num batches 81
Init, Train Loss: 1.8199, Dev Loss: 1.8199, Dev Loss confidence interval: 1.8124, 1.8274
Step 0, Train Loss: 1.7570, Dev Loss: 1.7550, Dev Loss confidence interval: 1.7460, 1.7640, LR: 0.001000
Step 10, Train Loss: 1.7571, Dev Loss: 1.7551, Dev Loss confidence interval: 1.7462, 1.7641, LR: 0.001000
Step 20, Train Loss: 1.7570, Dev Loss: 1.7550, Dev Loss confidence interval: 1.7461, 1.7640, LR: 0.001000
Step 30, Train Loss: 1.7569, Dev Loss: 1.7548, Dev Loss confidence interval: 1.7458, 1.7637, LR: 0.001000
Step 40, Train Loss: 1.7461, Dev Loss: 1.7386, Dev Loss confidence interval: 1.7296, 1.7476, LR: 0.001000
Step 50, Train Loss: 1.7247, Dev Loss: 1.7345, Dev Loss confidence interval: 1.7257, 1.7433, LR: 0.000500
Step 60, Train Loss: 1.7026, Dev Loss: 1.7485, Dev Loss confidence interval: 1.7397, 1.7573, LR: 0.000500
Step 70, Train Loss: 1.6924, Dev Loss: 1.7434, Dev Loss confidence interval: 1.7346, 1.

KeyboardInterrupt: 

In [4]:
weight_decay = 0.002
model = LSTMProbNNDist(context_window, StudentTHead(), device)
train_path = "Train_StudentT_Regularized/"
model, train_losses, dev_losses = train_model(model, train_X, train_Y, dev_X, dev_Y, lr, weight_decay, num_steps, batch_size=batch_size,
    device=device, output_folder=train_path, lr_decay_step=decay_step, lr_decay_gamma=decay_gamma)



Initializing LSTM with context_window=60, dist=StudentTHead, dof=3
num batches 81
Init, Train Loss: 1.8201, Dev Loss: 1.8201, Dev Loss confidence interval: 1.8126, 1.8276
Step 0, Train Loss: 1.7571, Dev Loss: 1.7552, Dev Loss confidence interval: 1.7463, 1.7641, LR: 0.001000
Step 10, Train Loss: 1.7570, Dev Loss: 1.7550, Dev Loss confidence interval: 1.7460, 1.7640, LR: 0.001000
Step 20, Train Loss: 1.7571, Dev Loss: 1.7551, Dev Loss confidence interval: 1.7462, 1.7640, LR: 0.001000
Step 30, Train Loss: 1.7570, Dev Loss: 1.7551, Dev Loss confidence interval: 1.7461, 1.7640, LR: 0.001000
Step 40, Train Loss: 1.7570, Dev Loss: 1.7550, Dev Loss confidence interval: 1.7461, 1.7640, LR: 0.001000
Step 50, Train Loss: 1.7570, Dev Loss: 1.7550, Dev Loss confidence interval: 1.7461, 1.7640, LR: 0.000500
Step 60, Train Loss: 1.7570, Dev Loss: 1.7550, Dev Loss confidence interval: 1.7461, 1.7640, LR: 0.000500
Step 70, Train Loss: 1.7570, Dev Loss: 1.7550, Dev Loss confidence interval: 1.7461, 1.

KeyboardInterrupt: 

In [ ]:
model = LSTMProbNNDist(context_window, SkewedStudentTHead(), device)
train_path = "Train_SkewedStudentT/"
model, train_losses, dev_losses = train_model(model, train_X, train_Y, dev_X, dev_Y, lr, weight_decay, num_steps, batch_size=batch_size, device=device, output_folder=train_path)

In [ ]:
dof = 16
model = LSTMProbNNDist(context_window, BLogistic(dof - 2, device), device)
train_path = "Train_BLogistic/"
model, train_losses, dev_losses = train_model(model, train_X, train_Y, dev_X, dev_Y, lr, weight_decay, num_steps, batch_size=batch_size, device=device, output_folder=train_path)